# Lección 10 - Agentes de IA en Producción

En esta lección aprenderás **patrones de producción** para agentes de IA usando el Microsoft Agent Framework (Python). Cubrimos:

- **Observabilidad** — añadiendo temporización y registro a las interacciones del agente
- **Evaluación** — usando un agente evaluador para calificar la calidad de las respuestas
- **Gestión de costos** — estrategias para la optimización de tokens y selección de modelos

El escenario es un **agente de viajes** que ayuda a los usuarios a planificar viajes, con monitoreo y evaluación añadidos.


## Configuración


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Consideraciones para Producción

Llevar agentes de IA de prototipos a producción requiere una atención cuidadosa a tres pilares:

1. **Observabilidad** — Necesitas visibilidad sobre lo que hace el agente, cuánto tiempo tarda y qué herramientas utiliza. Sin trazabilidad y registro, depurar problemas en producción es casi imposible.

2. **Evaluación** — Las verificaciones automáticas de calidad aseguran que las respuestas del agente se mantengan precisas, completas y útiles con el tiempo. Un agente evaluador puede calificar las respuestas según criterios definidos.

3. **Gestión de Costos** — El uso de tokens impacta directamente el costo. Estrategias como la optimización de prompts, la selección de modelos y el almacenamiento en caché ayudan a mantener los gastos bajo control sin sacrificar la calidad.


## Construyendo un Agente Observable

Definimos herramientas de viaje y envolvemos la llamada al agente con temporización para que podamos monitorear la latencia. En producción, se integraría con OpenTelemetry o un backend de trazabilidad similar.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Patrones de Evaluación

Un patrón común en producción es utilizar un segundo agente como **evaluador**. El evaluador puntúa la respuesta del agente principal contra criterios predefinidos como completitud, exactitud y utilidad.

Esto permite:
- Puertas de calidad automatizadas antes de que las respuestas lleguen a los usuarios
- Detección de regresiones cuando cambian los prompts o los modelos
- Monitoreo continuo del rendimiento del agente a lo largo del tiempo


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Estrategias de Gestión de Costos

Controlar los costos es fundamental para los agentes de IA en producción. Aquí están las estrategias clave:

| Estrategia | Descripción |
|---|---|
| **Optimización de prompts** | Mantener las instrucciones del sistema concisas. Eliminar contexto redundante para reducir los tokens de entrada. |
    "| **Selección de modelo** | Usar modelos más pequeños y baratos (p.ej. GPT-4.1-mini) para tareas sencillas como clasificación o extracción, y reservar modelos más grandes para razonamiento complejo. |\n",
| **Caché** | Almacenar en caché los resultados de herramientas y consultas frecuentes para evitar llamadas API redundantes. |
| **Presupuestos de tokens** | Establecer límites de `max_tokens` para evitar respuestas inesperadamente largas. |
| **Agrupamiento** | Agrupar múltiples consultas de usuario en una sola llamada API cuando sea posible. |

En la práctica, un enfoque escalonado funciona bien: dirigir solicitudes simples a un modelo rápido y económico, y escalar solo las consultas complejas a uno más capaz.


## Resumen

En esta lección aprendiste a:

1. **Agregar observabilidad** a las interacciones del agente con temporización y registro, sentando las bases para el rastreo y monitoreo.
2. **Evaluar las respuestas del agente** automáticamente usando un agente evaluador que califica la integridad, precisión y utilidad.
3. **Gestionar costos** mediante la optimización de prompts, selección de modelos, almacenamiento en caché y presupuestos de tokens.

Estos patrones de producción ayudan a asegurar que tus agentes de IA sean confiables, medibles y rentables a gran escala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
